# SOP OTDR Preprocessing Pipeline — EllaLink / SSU-A

| | |
|---|---|
| **Version** | 1.0 |
| **Status** | Public |
| **Author** | Miquel Masanas |
| **Contact** | miquel.1.masanas@nokia.com |
| **Date** | March 2026 |
| **Project** | Submerse |
| **Cable system** | EllaLink (Sines, Portugal — Fortaleza, Brazil) |

---

## Introduction

This notebook introduces the pre-processing pipeline for perturbation observations from Nokia's Seabed Sensing Unit A (SSU-A), deployed on the EllaLink trans-Atlantic cable. It is intended for Submerse project partners with backgrounds in both telecommunications and geophysics who are unfamiliar with the raw data structure or its physical interpretation. It provides a brief description of the measurement technology, step-by-step pre-processing from raw data cleaning to the derivation of physical observables, and documented examples of the supporting functions in `soplib.py`.

Scientific analysis — including comparison with ocean-bottom seismometer (OBS) data, event detection, and tidal signal interpretation — is covered in `02_sop_otdr_observations_case_study.ipynb`.

All credits of the development of SSU unit go to Pierre Mertz and his team, and Sumudu Edirisinghe.
This work is only possible thanks to EllaLink, as they provided their network as a testbed for the device.
This project has received funding from the European Union’s Horizon Europe research and innovation programme under grant agreement No 101095055 (Submerse).

### SSU-A as a source of SOP-OTDR

Nokia's Seabed Sensing Unit A (SSU-A) is a dual SOP/RF-OTDR interrogator that leverages the high-loss loopbacks (HLLBs) present in submarine cable repeaters to measure changes in the state of polarization (SOP) and radio-frequency phase (RF) at each cable span. Environmental perturbations (seismic strain, hydrostatic pressure variations, and temperature changes) alter the local birefringence of the fiber, rotating the SOP away from its resting position (Zhan et al. 2021, Mecozzi 2024). By tracking these rotations span by span, SSU-A operates as a  geophysical sensor array. The HLLB-based localization approach was first demonstrated by Costa et al. (2023) using an eigenvalue method requiring three orthogonal input polarization states. This device is currently using fiber-bragg grating reflections every span to generate the reflections, therefore it is reflectometry but most of its power is generated in discrete locations rather than distributed continuous points of the fiber. In that regard, the technology is akin to multispan Microwave Frequency Fiber Interferometry (MFFI) as the reflections generate concatenated loops of fiber.

The measurement principle is optical time-domain reflectometry (OTDR): a pulse is launched from the interrogator at Sines (Portugal) and travels along the cable, with a fraction of the light reflected back at each HLLB. Reflections are separated by their time of arrival, each corresponding to a different repeater span. The nominal EllaLink cable length of around 6200 km and the approximate speed of light in silica (around 2×10⁵ km/s) give a round-trip propagation time of  around 62 ms, bounding the maximum sampling rate to  around 16 Hz. The operational sampling rate of SSU-A is 13.5 Hz, providing sensitivity across the teleseismic band and into long-period oceanographic phenomena.

**Important constraint:** SSU-A interrogates the fiber with a single fixed input polarization state. Unlike transponder-based SOP methods (Mecozzi 2024), which reconstruct the full Jones matrix from multiple input states, this system recovers the output state after several reflections of a single polarization input state, therefore measuring output electrical field polarization rather than fully characterized fiber birefringence. Therefore, a perturbation at span *n* propagates to the SOP measurements of all downstream spans *i > n*. This cumulative effect is a fundamental limitation of the present deployment and motivates the processing choices described in this notebook.


## Background

This notebook assumes basic familiarity with optical fiber sensing and polarization optics. The primary direct measurement of SOP-OTDR from a single state of polarization input to the fiber are the Stokes parameters of the reflections. Readers new to these concepts may find the following resources useful before proceeding:

Damask, Polarization Optics in Telecommunications (2005), Chapter 1, 2, 3  

https://en.wikipedia.org/wiki/Stokes_parameters

Zhan et al. 2021

Mecozzi 2024

Costa et al. (2023)

### References (see also `references.bib`)

- Zhan et al. (2021) https://doi.org/10.1126/science.abe6648
- Mecozzi (2024) — verify DOI
- Costa et al. (2023) — verify venue/DOI
- Damask (2005) *Polarization Optics in Telecommunications*


## Imports


In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src" / "soplib.py").exists():
    REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import scienceplots
from scipy.stats import median_abs_deviation
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from src.case_studies import JAN2025_TAIWAN, APR2025_REYKJANES
from src.paths import DataPaths
from src.soplib import *

# CASE = JAN2025_TAIWAN          # raw preprocessing demo day
CASE = APR2025_REYKJANES

PATHS = DataPaths.from_cwd(REPO_ROOT, case=CASE)

%matplotlib inline
plt.style.use(['science', 'no-latex'])
plt.rcParams['figure.dpi']  = 100
plt.rcParams['savefig.dpi'] = 300

print('soplib loaded.')
print(f'Case: {CASE.label}')
print(f'Repo root: {REPO_ROOT}')



## Setup and Data Loading


### Metadata visualization


In [ ]:
print_hdf5_structure(str(PATHS.sop_hdf5_path(derotated=False)))

### Data Loading


In [ ]:
PATHS = DataPaths.from_cwd(REPO_ROOT, case=CASE)
HDF5_DIR     = PATHS.hdf5_dir
CATALOG_PATH = PATHS.catalog_csv

START = CASE.day
END   = CASE.day
DAY   = CASE.day
MAG_THRESHOLD = CASE.mag_threshold

print(f'Loading SOP data {START} to {END} ...')
SOP_raw = load_sops(HDF5_DIR, START, END)
print(f'Loaded {len(SOP_raw)} repeaters.')
print(f'Time range: {SOP_raw[list(SOP_raw.keys())[0]].index[[0,-1]]}')

catalogue = load_catalogue(str(CATALOG_PATH))
catalogue = catalogue[DAY:DAY]

print(f'Loaded {len(catalogue)} events.')
print(f'Median sample rate fs = {calculate_fs(SOP_raw[0].index)} Hz.')

catalogue.head()


## Data pre-processing


### Outlier Detection — Adaptive MAD Z-score on Stokes Derivative

Outliers are promintent in the SOP measurements, due to the limited reflection power and limited input power to the fiber. A rolling median absolute deviation (MAD) computation of the differentiated signal and a Z score discrimination has shown good performance for discarding values that are not meaningful physically.
We detect outliers as physically implausible jumps in the Stokes vector,
defined as the chord length between consecutive SOP positions:

$$|\Delta S_n| = \sqrt{(\Delta S_1^n)^2 + (\Delta S_2^n)^2 + (\Delta S_3^n)^2}$$

Rather than a fixed threshold, we use a **rolling MAD-based robust Z-score**:

$$Z_n = \frac{|\Delta S_n - \tilde{\Delta S}|}{1.4826 \cdot \text{MAD}(\Delta S)}$$

where $\tilde{\Delta S}$ is the rolling median, and alongside the MAD factor, they are computed over a rolling window.
The 1.4826 factor makes MAD a consistent estimator of $\sigma$ for Gaussian data.

**Key property:** the rolling MAD adapts to local activity level. During
seismic events the baseline velocity on the sphere is increased, so the
detector does (should) not flag real geophysical signal as outliers. We recognize that a geodesic variation is the true measure of variation, but the chord length proves to visually clean physically meaningless samples.


### Analysis parameters setting


In [ ]:
# -- Parameters -----------------------------------------------------------
# REPEATER_EXAMPLES = np.arange(1,51,1)   # adjust to valid keys in SOP_raw
REPEATER_EXAMPLES = [18, 25, 33]   # adjust to valid keys in SOP_raw

# Magnitude threshold: 
MAG_THRESHOLD = 5


# Cable geometry
CABLE_LENGTH_KM    = 6200
N_REPEATERS        = 82
detector_distances = np.linspace(0, CABLE_LENGTH_KM, N_REPEATERS)

# Processing parameters
params_derotation = {
    'derotation_window_duration_seconds': 500,
}

params_outliers = {
    "mad_factor": 4,
    "outlier_sliding_window": 120,
}

print(f'Repeater spacing: {CABLE_LENGTH_KM / (N_REPEATERS-1):.1f} km')
print(f'Events above M{MAG_THRESHOLD}: {len(filter_catalogue(catalogue, MAG_THRESHOLD))}')

### Timing analysis

The interpolation and decimation are needed above all for spectral domain analysis, because the timestamps are native to the measurement equipment and are not dependent on peripheral equipments, only the jitter and time drift of the equipment itself affects the time axis. The equipment experiences typically 1 sample time delay every 4 samples as it flushes its buffer, which introduces frequency artifacts and shifts in the spectral domain, but does not affect time-domain waveforms. First, a overview of the timing performance follows:  

In [ ]:
# ── Sampling regularity analysis ─────────────────────────────────────────
rep = REPEATER_EXAMPLES[0]
df  = SOP_raw[rep]

t_raw  = (df.index - df.index[0]).total_seconds().values
dt_all = np.diff(t_raw)
dt_med = np.median(dt_all)

print(f"Nominal dt:   {dt_med*1000:.2f} ms  ({1/dt_med:.2f} Hz)")
print(f"Mean dt:      {dt_all.mean()*1000:.2f} ms")
print(f"Std dt:       {dt_all.std()*1000:.2f} ms")
print(f"Max dt:       {dt_all.max()*1000:.2f} ms  ({dt_all.max()/dt_med:.1f}x nominal)")
print(f"Min dt:       {dt_all.min()*1000:.2f} ms")
print(f"Gaps > 3x dt: {(dt_all > dt_med*3).sum()}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
fig.suptitle(f'Sampling Regularity, Repeater {rep}', fontsize=12)

# Histogram of inte, sample intervals
axes[0].hist(dt_all * 1000, bins=100, color='steelblue', edgecolor='none',density=True)
axes[0].axvline(dt_med * 1000, color='red', linewidth=1.2,
                linestyle='--', label=f'median {dt_med*1000:.1f}ms')
axes[0].set_xlabel('Inter-sample interval (ms)')
axes[0].set_ylabel('Count')
axes[0].set_title('Time step distribution')
axes[0].legend()

# Gap timeline, where are the large gaps?
gap_times = t_raw[:-1]
gap_sizes = dt_all
axes[1].scatter(gap_times / 3600, gap_sizes, 
                s=6, color='red', alpha=0.7)
axes[1].axhline(dt_med * 3, color='gray', linewidth=0.8,
                linestyle='--', label='3x nominal dt')
axes[1].set_xlabel('Time from start (hours)')
axes[1].set_ylabel('Gap size (s)')
axes[1].set_title('Gap locations and sizes')
axes[1].legend()

plt.tight_layout()
plt.show()

The inter-sample interval distribution shows discrete bands at 1x, 2x, and occasionally 3x the nominal dt (~74ms), consistent with a buffered packet transmission system. The 2x band is ubiquitous and represents normal double-buffering. Gaps exceeding 3x nominal dt (~222ms) are treated as genuine dropouts and interpolated linearly to avoid Runge oscillations at gap boundaries. The decimation is done here to alleviate the load of the long time span processing in this example, and to filter high frequency noise. It is applied with the default 'iir' filter of scipy.signal.decimate, an order 8 Chebyshev type, with zero_phase set to true.

In [ ]:
# Step 0 - Clean outliers (mask from repeater 0, applied globally)
print('Cleaning outliers ...')
_, outlier_mask = clean_outliers_SOP(SOP_raw[0], params_outliers)
flat_mask = outlier_mask.flatten()

# %% 

# 2. Use a dictionary comprehension to apply the mask across all keys.
SOP_cleaned = {
    ID: SOP_raw[ID].loc[~flat_mask] 
    for ID in REPEATER_EXAMPLES
}

### Decimation and interpolation


In [ ]:

dt = get_dt(SOP_cleaned[REPEATER_EXAMPLES[0]])

# Step 1 - Decimate (13.51 → 3.38Hz with q=4, or adjust to frequency band of analysis)
q=4
large_gap_threshold_samples = 3

print(f"Original fs = {1/dt} Hz, new fs = {1/(dt*q)}")

print('Interpolating and decimating ...')

SOP_decimated = {}

for key, df in SOP_cleaned.items():
    # Create a copy so you don't mutate the original data    
    SOP_decimated[key] = resample_and_decimate(df, q=q, large_gap_threshold_samples=large_gap_threshold_samples)

print('Interpolation and decimation completed')



In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
fig.suptitle(f'Cleaning and Resampling Repeater {REPEATER_EXAMPLES[0]} {DAY}', fontsize=12)

raw = SOP_raw[REPEATER_EXAMPLES[0]][DAY:DAY]
cln = SOP_cleaned[REPEATER_EXAMPLES[0]][DAY:DAY]
dec = SOP_decimated[REPEATER_EXAMPLES[0]][DAY:DAY]

for ax, col in zip(axes, ['S1', 'S2', 'S3']):
    ax.plot(raw.index, raw[col], linewidth=0.3, color='lightcoral',
            label='raw', alpha=0.6)
    ax.plot(cln.index, cln[col], linewidth=0.3, color='steelblue',
            label='cleaned', alpha=0.8)
    ax.plot(dec.index, dec[col],marker='.', markersize=3, linewidth=0.0, color='darkgreen',
            label='decimated', alpha=0.5)
    ax.set_ylabel(col)
    ax.legend(fontsize=8, loc='upper right')

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

### Normalization and de-rotation

SSU-A does not have polarization diversity interrogation, unlike transponder-based SOP measurements — it measures field rotation in the
fiber rather than the rotation vector itself. As a result, larger directional swings are expected. We also observe power and DOP variations that require normalization to the Poincaré sphere (DOP = 1):

$$\hat{S}_n = \frac{[S_1, S_2, S_3]_n}{\sqrt{S_1^2 + S_2^2 + S_3^2}}$$


De-rotation removes the slow polarization drift (thermal, tidal) by applying a rotation that maps the local mean SOP to the north pole of the Poincaré sphere, leaving only the fast residual dynamics.
The local mean is computed as the Fréchet mean on S² — the point that minimizes the sum of squared geodesic distances to all vectors in a rolling window:

$$\mu^* = \arg\min_{\mu \in S^2} \sum_{i} d_g(S_i,\, \mu)^2$$

The Euclidean mean is not adequate here because it does not lie on the sphere and introduces a systematic bias for large angular swings, which the system exhibits. The Fréchet mean is found iteratively via Riemannian gradient descent: at each step, window vectors are projected onto the tangent plane at the current estimate (Log map), averaged in the flat tangent space, then mapped back to the sphere (Exp map). Convergence is typically achieved in 3–5 iterations.

In [ ]:

# Step 2 - Normalize to Poincaré sphere (DOP = 1)
print('Normalizing ...')

SOP_normalized = {}

for key, df in SOP_decimated.items():
    # Create a copy so you don't mutate the original data
    df_norm = df.copy()
    
    # Calculate the denominator
    mag = np.sqrt(df[['S1', 'S2', 'S3']].pow(2).sum(axis=1))
    
    # Update only the specific columns
    df_norm[['S1', 'S2', 'S3']] = df[['S1', 'S2', 'S3']].div(mag, axis=0)
    
    SOP_normalized[key] = df_norm


In [ ]:
plot_poincare_plotly(SOP_normalized[REPEATER_EXAMPLES[1]])

**Beware! De-rotation can take close to 3 minutes per day per repeater at 3 Hz approx (computer-dependent), parallelization is recommended, sequential de-rotation is mainly good for exploration and examples**

In [ ]:

# Step 3 - Derotate (Fréchet rolling mean on S²)
print('Derotating ...')

SOP_derotated = {}
SOP_derotated_slow = {}

for key, df in SOP_normalized.items():
    print(f'Processing repeater num {key} of list {(SOP_normalized.keys())}.')
    SOP_derotated[key],SOP_derotated_slow[key] = derotate_df(key, df, params_derotation)
    print(f'Repeater {key} finished.')

print(f'Processed {len(SOP_derotated)} repeaters.')


In [ ]:
plot_poincare_plotly(SOP_derotated[REPEATER_EXAMPLES[1]])

### Catalog vs processed data


In [ ]:

fig, axes = plt.subplots(len(REPEATER_EXAMPLES), 3, figsize=(10, 3*len(REPEATER_EXAMPLES)),
                         sharex=True)
fig.suptitle(f"Raw Stokes Parameters — {DAY}", fontsize=13)

for row, rep_id in enumerate(REPEATER_EXAMPLES):
    df = SOP_derotated[rep_id]
    day_df = df[DAY:DAY]
    catalogue = catalogue[DAY:DAY]
    for col, stokes in enumerate(['S1', 'S2', 'S3']):
        ax = axes[row, col]
        ax.plot(day_df.index, day_df[stokes], linewidth=0.5, color='steelblue')
        add_event_lines(ax, catalogue, MAG_THRESHOLD)
        ax.set_ylabel(f"Rep {rep_id}" if col == 0 else "")
        if row == 0:
            ax.set_title(stokes)
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))

fig.autofmt_xdate()
plt.show()

## Derived Quantities
All derived quantities are pre-computed here for all repeaters and stored
in `df_derived`, a dict with the same repeater keys as `SOP_derotated`.
Each value is a DataFrame with columns:

| Column | Description | Units |
|---|---|---|
| `omega` | Geodesic angular rate between consecutive SOP vectors, signed | rad/s |
| `speed_std` | Rolling std of ω over `ROLLING_WINDOW` | rad/s |
| `stokes_std` | Addition of rolling variance of $s_{i}$ over `ROLLING_WINDOW`, root-squared | a.u |
| `kurtosis` | Rolling kurtosis of ω — impulsive vs sustained discrimination | dimensionless |


Explanations follow their computation


In [ ]:
# ── 2.0 Compute derived quantities ────────────────────────────────────────
print('Computing derived quantities ...')

highcut = 1
ROLLING_WINDOW = '20s'
df_derived = {}

for key, df in SOP_derotated.items():
    S   = df[['S1', 'S2', 'S3']].dropna().to_numpy()
    dt  = get_dt(df)
    fs = 1/dt
    S_smooth = np.column_stack([
        lowpass_filter_sos(S[:, i], highcut=highcut, fs=fs, order=4)
        for i in range(3)
    ])
    
    
    # Geodesic angular rate
    omega_filt = sop_angular_rate_signed(S_smooth, dt)
    omega_s_filt = pd.Series(omega_filt, index=df.index[:-1], name='omega_filt')
    omega = sop_angular_rate_signed(S, dt)
    omega_s = pd.Series(omega, index=df.index[:-1], name='omega')
    # Rolling statistics
    roll_std  = np.sqrt(omega_s.rolling(ROLLING_WINDOW).var())
    roll_kurt = omega_s.rolling(ROLLING_WINDOW).kurt()
    roll_std_s = np.sqrt(df[['S1','S2','S3']].rolling(ROLLING_WINDOW).var().sum(axis=1).to_numpy()[1::]) # we need to loose one sample as the angular rate is a differentiation and the first sample is undefined

    df_derived[key] = pd.DataFrame({
        'omega': omega_s,
        'omega_filtered': omega_s_filt,
        'speed_std': roll_std,
        'stokes_std': roll_std_s,
        'kurtosis': roll_kurt,
    })

print(f'Derived quantities computed for {len(df_derived)} repeaters.')
print(f'Columns: {list(df_derived[list(df_derived.keys())[0]].columns)}')

### Angular rate of variation
Rather than the Euclidean difference between consecutive Stokes vectors,
we compute the true geodesic angle on the Poincaré sphere:

$$\omega_n = sng((v_{n} \times v_{n+1})\cdot \hat{n}) \frac{\arctan2(||v_n \times v_{n+1}||,\ v_n \cdot v_{n+1})}{\Delta t}$$

This is robust for large polarization swings where the Euclidean
approximation breaks down. Implementation in soplib.sop_angular_rate. In the following plots, it is apparent that angular speed is sensitive to the environment effects, but different repeaters show different signal to noise performances and shape of the effects. As a sign reference for the angular speed, we took 
$$\hat{n} = [0,0,1]^T$$

The computation of the rate of variation is a kind of numerical differentiation that amplifies high-frequency noise, as can be observed from the power spectral densities.


#### Time domain angluar speed
First we plot the time-domain angular velocity and its filtered version.

In [ ]:
fig, axes = plt.subplots(len(REPEATER_EXAMPLES[::1]), 1,
                         figsize=(10, 2.5*len(REPEATER_EXAMPLES[::1])),
                         sharex=True)
fig.suptitle(f'Geodesic Angular Rate ω — {DAY}', fontsize=12)

events_filtered    = filter_catalogue(catalogue[DAY:DAY], MAG_THRESHOLD)
LABEL_THRESHOLD = f'Minimum magnitude = {MAG_THRESHOLD}'

for ax, key in zip(axes, REPEATER_EXAMPLES[::1]):
    omega_day = df_derived[key]['omega'][DAY:DAY]
    omega_day_filt = df_derived[key]['omega_filtered'][DAY:DAY]
    ax.plot(omega_day.index, omega_day.values,
            linewidth=0.4, color='steelblue', alpha=0.6)
#     fs = calculate_fs(omega_day.index)
#     omega_day_filtered = bandpass_filter_sos(omega_day.values, lowcut = 0.005, highcut= 0.7, fs=fs, order=4)
    ax.plot(omega_day_filt.index, omega_day_filt,
            linewidth=1, color='steelblue', alpha=1)
    ax.set_ylabel(f'Rep {key}\n(rad/s)', fontsize=8)
    add_event_lines(ax, events_filtered,
                    mag_min=MAG_THRESHOLD)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))

axes[-1].set_xlabel('Time (UTC)')
fig.autofmt_xdate()
plt.tight_layout()
plt.show()


#### Angular velocity power spectral density compared to raw metrics

Now we observe the power spectral density for the processing pipeline

In [ ]:

fig, axes = plt.subplots(len(REPEATER_EXAMPLES[::1]), 1,
                         figsize=(10, 4*len(REPEATER_EXAMPLES[::2])), sharex=True)

# Reference period lines
period_markers = {
    '24h':  1/86400,
    '12h':  1/43200,
    '1h':   1/3600,
    '10min':1/600,
    '1min': 1/60,
}

for ax, rep_id in zip(axes, REPEATER_EXAMPLES):
    # Unfiltered
    df_raw = SOP_cleaned[rep_id]
    dt = get_dt(df_raw)
    fs = 1 / dt
    x_raw = df_raw['S1'].dropna().values + df_raw['S2'].dropna().values + df_raw['S3'].dropna().values
    f_raw, p_raw = psd_db(x_raw, fs)

    ax.plot(f_raw, p_raw, color='gray', linewidth=1.0,
            alpha=0.7, label='unfiltered')

# Derotated
    df_der = SOP_derotated[rep_id]
    x_der = df_der['S1'].dropna().values + df_der['S2'].dropna().values + df_der['S3'].dropna().values
    dt = get_dt(df_der)
    fs = 1 / dt
    f_der, p_der = psd_db(x_der, fs)
    p_der_smooth = uniform_filter1d(p_der, size=20)

    ax.plot(f_der, p_der, color='steelblue', linewidth=1.0,
            label='derotated')
    

# Angular speed
    df_speed = df_derived[rep_id]
    x_der = df_speed['omega'].dropna().values
    dt = get_dt(df_speed)
    fs = 1 / dt
    f_omega, p_omega = psd_db(x_der, fs)

    ax.plot(f_omega, p_omega, color='sandybrown', linewidth=1.0,
            label='angular speed')
    
# Angular speed filtered
    df_speed = df_derived[rep_id]
    x_der = df_speed['omega_filtered'].dropna().values
    dt = get_dt(df_speed)
    fs = 1 / dt
    f_omega, p_omega = psd_db(x_der, fs)

    ax.plot(f_omega, p_omega, color='lightcoral', linewidth=1.0,
            label='angular speed filtered')

# Period markers
    for label, freq in period_markers.items():
        ax.axvline(freq, color='red', linewidth=0.6, linestyle=':', alpha=0.6)
        ax.text(freq, ax.get_ylim()[0] + 2, label,
                color='red', fontsize=18, rotation=90, va='bottom')

    ax.set_xscale('log')
    ax.set_ylabel("dB / Hz")
    ax.set_title(f"Repeater {rep_id}")
    ax.legend(fontsize=8)

axes[-1].set_xlabel("Frequency (Hz)")
plt.tight_layout()
plt.show()

### Signal Power and Impulsivity — Variance and Kurtosis

We can investigate the variance or standard deviation of both the de-rotated stokes vectors, by computing them by each dimension and adding them 

$$\sigma_{stokes, j}^2 = \sum_{i=1,2,3}{Var_{N}(s_{i,j})}$$

what we called stokes_variance. 
We can also compute the variance or std of $\omega$ providing an estimate of local-time signal power

$$\sigma_{\omega, j}^2 = \sum_{i=1,2,3}{Var_{N}(\omega_{j})}$$


Since after de-rotation $\bar{\omega} \approx 0$, variance reduces to mean square angular velocity — directly analogous to signal power in the angular domain.

The rolling excess kurtosis discriminates between sustained and impulsive perturbations:

$$\kappa(t) = \frac{\mu_4}{\sigma^4} - 3$$

where $\mu_4$ is the fourth central moment. For a Gaussian process $\kappa = 0$. Seismic body wave arrivals (P, S) are characteristically impulsive ($\kappa \gg 0$), while sustained perturbations such as ocean swell or continuous tremor remain near-Gaussian ($\kappa \approx 0$).

Kurtosis can be used in seismological signal processing as a characteristic function for phase detection. While its implementation and use for discrimination is beyond the expertise of optical telecom engineers, we include its computation here and a big picture approach on its interpretation, without implying accurate implementation or analysis based on it.

Latorre et al. (2025) (https://doi.org/10.1093/gji/ggaf136) developed a multiband kurtosis-based phase picker validated on DAS data from submarine fibre-optic cables and OBS recordings , directly analogous to our deployment. Saragiotis et al. (2004) introduced automatic P-phase picking using maximum kurtosis (https://doi.org/10.1109/TGRS.2002.800438). Baillard et al. (2014) presented an automatic kurtosis-based P and S phase picker validated on ocean-bottom seismometer data 
([doi:10.1785/0120120347](https://doi.org/10.1785/0120120347))


**Interpretation guide:**

| Variance | Kurtosis | Interpretation |
|---|---|---|
| High | Moderate (~0-3) | Sustained event — seismic surface waves, storm |
| Low | High (>5) | Impulsive transient — P/S body wave arrival, ship |
| High | High | Energetic impulsive event — large nearby earthquake |
| Low | Low | Quiet background |





In [ ]:
# --- Configuration ---
rep = REPEATER_EXAMPLES[1]
dt = get_dt(SOP_derotated[rep])
window_size = '60s'

# Calculate smoothed versions
stokes_smooth = df_derived[rep]['stokes_std'].rolling(window=window_size, center=True).mean()
speed_smooth = df_derived[rep]['speed_std'].rolling(window=window_size, center=True).mean()

fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

# Top Plot: Stokes Standard Deviation
axes[0].plot(df_derived[rep].index, df_derived[rep]['stokes_std'], 
             linewidth=0.5, color='steelblue', alpha=0.6, label='Raw stokes std')
axes[0].plot(stokes_smooth.index, stokes_smooth, 
             linewidth=1, color='midnightblue', label=f'Smoothed (N={window_size})')
axes[0].set_ylabel('angular speed (rad/s)')
axes[0].legend(loc='upper right')
add_event_lines(axes[0], events_filtered,
                    mag_min=MAG_THRESHOLD)

# Bottom Plot: Speed Standard Deviation
axes[1].plot(df_derived[rep].index, df_derived[rep]['speed_std'], 
             linewidth=0.5, color='darkorange', alpha=0.6, label='Raw speed std')
axes[1].plot(speed_smooth.index, speed_smooth, 
             linewidth=1, color='tab:red', label=f'Smoothed (N={window_size})')
axes[1].set_ylabel('angular speed (rad/s)')
axes[1].legend(loc='upper right')
add_event_lines(axes[1], events_filtered,
                    mag_min=MAG_THRESHOLD)

# Formatting
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
fig.autofmt_xdate()
axes[0].set_title(f'Repeater {rep}: Angular Variation and Speed Dynamics')
plt.tight_layout()
plt.show()

Now we take a look at the kurtosis complementing the standard deviation, on top of the original signal, where we see that the std correclty approximates the signal envelope

In [ ]:
# --- Configuration ---
rep = REPEATER_EXAMPLES[1]
dt = get_dt(SOP_derotated[rep])
window_size_kurtosis = '30s'
window_size_std = '30s'
# Calculate smoothed versions
std_smooth = df_derived[rep]['speed_std'].rolling(window=window_size_std, center=True).mean()
kurtosis_smooth = df_derived[rep]['kurtosis'].rolling(window=window_size_kurtosis, center=True).mean()
fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

# Top Plot: Stokes Standard Deviation
axes[0].plot(df_derived[rep].index, df_derived[rep]['omega'], linewidth=0.5, color='darkorange', alpha=0.4, label='Raw speed')
axes[0].plot(df_derived[rep].index, std_smooth, linewidth=1, color='red', alpha=1, label='Smoothed std window N={window_size_std}s')
axes[0].set_ylabel('angular speed standard deviation (rad/s)')
axes[0].legend(loc='upper right')
add_event_lines(axes[0], events_filtered,mag_min=MAG_THRESHOLD)

# Bottom Plot: kurtosis
axes[1].plot(df_derived[rep].index, df_derived[rep]['kurtosis'], linewidth=0.5, color='steelblue', alpha=0.6, label='Raw kurtosis')
axes[1].plot(kurtosis_smooth.index, kurtosis_smooth, linewidth=1, color='tab:blue', label=f'Smoothed kurtosis window N={window_size_kurtosis}s')
axes[1].set_ylabel('kurtosis')
axes[1].legend(loc='upper right')
add_event_lines(axes[1], events_filtered,mag_min=MAG_THRESHOLD)

# Formatting
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
fig.autofmt_xdate()
axes[0].set_title(f'Repeater {rep}: Angular Variation and Speed Dynamics')
plt.tight_layout()
plt.show()

---
## Spatial Picture — Where activation occurs along the cable

Colorplot of Stokes variance across all repeaters vs time. As mentioned before, SOP OTDR coming from SSU-A does not provide the full SOP rotation transformation of the fiber, but rather how a single input state of polarization is rotated. Therefore, by not having a mechanism on how to undo cumulative effects, when repeater $n$ is perturbed we expect activation for all repeaters $i> n$ instantly as the light propagates through the link.

Nevertheless, colorplots showcase repeaters with excessive continuous power levels and noise for data cleaning.

Colorplots can show that the repeaters tend to respond without delay and showcase the localized continuous noise in certain repeaters, as well as the increasing noise as the link advances.

**For this plot to be informative, all repeaters should be processed.** The tutorial cells above only run the pipeline on a few example repeaters; here we load the pre-made `*_derotated.h5` file shipped with the dataset (full cable, already cleaned/decimated/de-rotated).


In [ ]:
# Load pre-made de-rotated SOP for all repeaters (see PATHS.sop_hdf5_path(derotated=True))
print(f'Loading pre-processed SOP data from {PATHS.sop_hdf5_path(derotated=True).name} ...')
SOP_derotated_full = load_sops(HDF5_DIR, START, END, suffix='_derotated')
print(f'Loaded {len(SOP_derotated_full)} repeaters.')

# Derived quantities on the full cable (same recipe as above)
print('Computing derived quantities for all repeaters ...')
df_derived_full = {}

for key, df in SOP_derotated_full.items():
    S = df[['S1', 'S2', 'S3']].dropna().to_numpy()
    dt = get_dt(df)
    fs = 1 / dt
    S_smooth = np.column_stack([
        lowpass_filter_sos(S[:, i], highcut=highcut, fs=fs, order=4)
        for i in range(3)
    ])

    omega_filt = sop_angular_rate_signed(S_smooth, dt)
    omega_s_filt = pd.Series(omega_filt, index=df.index[:-1], name='omega_filt')
    omega = sop_angular_rate_signed(S, dt)
    omega_s = pd.Series(omega, index=df.index[:-1], name='omega')

    roll_std = np.sqrt(omega_s.rolling(ROLLING_WINDOW).var())
    roll_kurt = omega_s.rolling(ROLLING_WINDOW).kurt()
    roll_std_s = np.sqrt(
        df[['S1', 'S2', 'S3']].rolling(ROLLING_WINDOW).var().sum(axis=1).to_numpy()[1::]
    )

    df_derived_full[key] = pd.DataFrame({
        'omega': omega_s,
        'omega_filtered': omega_s_filt,
        'speed_std': roll_std,
        'stokes_std': roll_std_s,
        'kurtosis': roll_kurt,
    })

print(f'Derived quantities computed for {len(df_derived_full)} repeaters.')

SOP_power_full = {
    k: pd.Series(df['S1']**2 + df['S2']**2, index=df.index)
    for k, df in SOP_derotated_full.items()
}

plot_repeater_colormap(
    SOP_power_full,
    title='SOP power (all repeaters)',
    cbar_label='',
    cmap='inferno',
    db_scale=True,
    catalogue=catalogue,
    mag_min=MAG_THRESHOLD,
)
plt.show()

plot_repeater_colormap(
    df_derived_full,
    quantity='speed_std',
    title='std of omega (all repeaters)',
    cbar_label='',
    cmap='inferno',
    db_scale=True,
    catalogue=catalogue,
    mag_min=MAG_THRESHOLD,
)
plt.show()


---
## Time-Frequency Evolution — Spectrogram

Reveals transient energy bursts at specific frequency bands (e.g. seismic surface waves).

**Time span:** 1 day.  
**Period range:** 1s to 1 day.

In [ ]:


rep_id = REPEATER_EXAMPLES[1]
df = SOP_cleaned[rep_id]
day_df = df[DAY:DAY].dropna()
dt = get_dt(df)
fs = 1 / dt

freqSpan = [1*1e-4, highcut]
windowSize = 2**10

spectrogramData_1, _ , _ = generateSpectrogram_scipy(day_df['S1'].values, day_df.index, windowSize, freqSpan, overlapPercentage=0.85,nfft_size = 2**13)
spectrogramData_2, timeAxis, frequencyAxis = generateSpectrogram_scipy(day_df['S2'].values, day_df.index, windowSize, freqSpan, overlapPercentage=0.85,nfft_size = 2**13)

spectrogramData = 10*np.log10(spectrogramData_1 + spectrogramData_2 + 1e-12)


repeaterID = f'Repeater_{rep_id}'
plot_spectrogram_results(spectrogramData, timeAxis, frequencyAxis, repeaterID,save=False, path = "",cmap="magma")
plot_spectrogram_results(spectrogramData, timeAxis, frequencyAxis, repeaterID,save=False, path = "",logplot=True,cmap="magma")


In the following code, we take a look on how the several processing steps show in an spectrogram

In [ ]:
# Select one day of data
rep_id = 1


processing_step_list = [SOP_cleaned[REPEATER_EXAMPLES[rep_id]],SOP_decimated[REPEATER_EXAMPLES[rep_id]],SOP_derotated[REPEATER_EXAMPLES[rep_id]],df_derived[REPEATER_EXAMPLES[rep_id]],df_derived[REPEATER_EXAMPLES[rep_id]]]
processing_step_list_names = ['Cleaned','Decimated','Derotated','Omega','Omega filtered']


for df, name in zip(processing_step_list,processing_step_list_names):
    day_df = df[DAY:DAY]
    if name == 'Omega':
        spectrogramData, timeAxis, frequencyAxis = generateSpectrogram_scipy(day_df['omega'].values, day_df.index, windowSize, freqSpan, overlapPercentage=0.85,nfft_size = 2**13)
        spectrogramData = 10*np.log10(spectrogramData + 1e-12)
    
    elif name=='Omega filtered':
        spectrogramData, timeAxis, frequencyAxis = generateSpectrogram_scipy(day_df['omega_filtered'].values, day_df.index, windowSize, freqSpan, overlapPercentage=0.85,nfft_size = 2**13)
        spectrogramData = 10*np.log10(spectrogramData + 1e-12)
    
    else:
        spectrogramData, timeAxis, frequencyAxis= generateSpectrogram_scipy(day_df['S1'].values + day_df['S2'].values, day_df.index, windowSize, freqSpan, overlapPercentage=0.85,nfft_size = 2**13)
        spectrogramData = 10*np.log10(spectrogramData + 1e-12)


    repeaterID = f'Repeater_{REPEATER_EXAMPLES[rep_id]} ' + name
    plot_spectrogram_results(spectrogramData, timeAxis, frequencyAxis, repeaterID,save=False, path = "",logplot=True,cmap="magma")



## Conclusions

This notebook documents the SSU-A SOP-OTDR preprocessing pipeline for the EllaLink cable, producing the following data products from raw Stokes measurements:
- `SOP_derotated` — cleaned, decimated, normalized, and de-rotated Stokes vectors on the Poincaré sphere
- `df_derived` — derived physical observables per repeater: geodesic angular rate (ω), rolling angular speed variance, Stokes variance, and kurtosis

Pipeline outputs have been visually validated against the USGS earthquake catalogue and show physically consistent responses to teleseismic events. Tidal signals visible in the raw phase data provide an additional independent sanity check.

Companion notebooks covering the RF-OTDR phase channel (SSU-A) and transponder-based SOP sensing (Nokia's CHM6 coherent transceiver) follow the same structure. Scientific analysis including comparison with ocean-bottom seismometer (OBS) recordings and standard seismic stations is provided in a separate results notebook.

---
## Event catalogues used:

- [iris](https://ds.iris.edu/wilber3/find_event) — Seismological Facility for the Advancement of Geoscience (SAGE)
- [USGS](https://earthquake.usgs.gov/earthquakes/search/) — global coverage, The one used in this notebook


## Other notebook utils


In [ ]:
# Update library

import importlib
import src.soplib
importlib.reload(src.soplib)
from src.soplib import *